In [1]:
!pip install -q langchain-text-splitters chromadb sentence-transformers groq google-genai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [8]:
!pip install -q groq

In [10]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

client = Groq(api_key=GROQ_API_KEY)

print("Groq client initialized successfully!")

Groq client initialized successfully!


In [2]:
import os
import getpass

def get_key(name):
    key = os.environ.get(name, "")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get(name) or ""
        except Exception:
            key = ""
    if not key:
        key = getpass.getpass(f"Enter {name} (press Enter to skip): ").strip()
    return key

GROQ_API_KEY = get_key("GROQ_API_KEY")
GEMINI_API_KEY = get_key("GEMINI_API_KEY")

print("Groq key provided:  ", bool(GROQ_API_KEY))
print("Gemini key provided:", bool(GEMINI_API_KEY))

if not GROQ_API_KEY and not GEMINI_API_KEY:
    print("\nNo keys provided. The retrieval parts will still run, but LLM answers will be skipped.")


Enter GROQ_API_KEY (press Enter to skip): ··········
Enter GEMINI_API_KEY (press Enter to skip): ··········
Groq key provided:   True
Gemini key provided: True


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

documents = {
    "ai_overview.txt": """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
""",
    "embeddings_in_practice.txt": """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once.
""",
    "rag_applications.txt": """Practical Applications of Retrieval-Augmented Generation

RAG is useful for internal knowledge assistants, customer support chatbots, and document question-answering tools. It retrieves relevant passages from a trusted collection before a language model writes an answer.
For example, a university assistant can retrieve course policies from official documents, while a support bot can find troubleshooting steps from product manuals. The answer should stay grounded in the retrieved passages and identify the sources used.
"""
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250, chunk_overlap=40, separators=["\n\n", "\n", ". ", " ", ""]
)

client = chromadb.Client()
hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = client.create_collection(name="rag_chunks", embedding_function=hf_embedder)

for source, text in documents.items():
    doc_chunks = splitter.split_text(text)
    collection.add(
        documents=doc_chunks,
        metadatas=[{"source": source, "chunk_index": i} for i in range(len(doc_chunks))],
        ids=[f"{source}_chunk_{i}" for i in range(len(doc_chunks))]
    )

print("Total chunks stored:", collection.count())


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total chunks stored: 17


In [4]:
def retrieve_context(query, n_results=4, similarity_threshold=0.25):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    context_chunks = []
    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        similarity = 1 - distance
        if similarity >= similarity_threshold:
            context_chunks.append({
                "text": doc,
                "source": meta["source"],
                "chunk_index": meta["chunk_index"],
                "similarity": round(similarity, 4)
            })
    return context_chunks

# Quick check that retrieval works
test_chunks = retrieve_context("What is RAG?")
print("Chunks retrieved:", len(test_chunks))
for c in test_chunks:
    print(f"  [{c['similarity']}] {c['source']} #{c['chunk_index']}")


Chunks retrieved: 2
  [0.7343] rag_applications.txt #1
  [0.5667] ai_overview.txt #6


In [11]:
SYSTEM_PROMPT = (
    "You are a helpful assistant that answers using ONLY the provided context. "
    "If the answer is absent, reply exactly: \"I don't know based on the provided documents.\" "
    "For questions answered from context, format the response as exactly 3 bullet points. "
    "Keep each bullet concise and cite the source file name(s) used."
)
RAG_TEMPERATURE = 0.1

def build_user_prompt(query, context_chunks):
    context_text = "\n\n".join(
        f"[Source: {c['source']}, chunk {c['chunk_index']}]\n{c['text']}"
        for c in context_chunks
    )
    return f"Context:\n{context_text}\n\nQuestion: {query}\n\nAnswer:"

sample_chunks = retrieve_context("How does RAG improve a language model?")
print(build_user_prompt("How does RAG improve a language model?", sample_chunks))


Context:
[Source: ai_overview.txt, chunk 6]
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG

[Source: rag_applications.txt, chunk 1]
RAG is useful for internal knowledge assistants, customer support chatbots, and document question-answering tools. It retrieves relevant passages from a trusted collection before a language model writes an answer.

[Source: ai_overview.txt, chunk 2]
understanding language, recognizing images, making decisions, and learning from experience.

Question: How does RAG improve a language model?

Answer:


In [12]:
from groq import Groq

GROQ_MODELS = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant", "openai/gpt-oss-20b"]

def ask_groq(user_prompt):
    if not GROQ_API_KEY:
        return "[Groq skipped: no API key provided]"

    groq_client = Groq(api_key=GROQ_API_KEY)
    last_error = None

    for model_name in GROQ_MODELS:
        try:
            response = groq_client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=RAG_TEMPERATURE,
                max_tokens=600
            )
            answer = response.choices[0].message.content
            if answer:
                return answer.strip()
        except Exception as e:
            last_error = e

    return f"[Groq error: {last_error}]"

question = "How does RAG improve a language model?"
chunks = retrieve_context(question)
print("Groq answer:\n")
print(ask_groq(build_user_prompt(question, chunks)))


Groq answer:

- Combines a search system with a language model, allowing the model to pull in up‑to‑date information rather than relying solely on memorized training data (ai_overview.txt).  
- Retrieves relevant passages from a trusted collection before generating a response, improving accuracy and relevance (rag_applications.txt).  
- Enables the model to answer questions in domains such as internal knowledge assistants, customer support, and document QA with higher confidence (rag_applications.txt).


## 5. Calling Google Gemini

We use the official `google-genai` SDK. Again, a short model list gives us an automatic fallback.
`system_instruction` plays the same role as the system message we used with Groq.


In [13]:
from google import genai
from google.genai import types

GEMINI_MODELS = ["gemini-2.5-flash", "gemini-3-flash-preview", "gemini-2.5-flash-lite"]

def ask_gemini(user_prompt):
    if not GEMINI_API_KEY:
        return "[Gemini skipped: no API key provided]"

    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    last_error = None

    for model_name in GEMINI_MODELS:
        try:
            response = gemini_client.models.generate_content(
                model=model_name,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=RAG_TEMPERATURE,
                    max_output_tokens=1500
                )
            )
            if response.text:
                return response.text.strip()
        except Exception as e:
            last_error = e

    return f"[Gemini error: {last_error}]"

print("Gemini answer:\n")
print(ask_gemini(build_user_prompt(question, chunks)))


Gemini answer:

* RAG combines a search system with a language model to enhance its performance (ai_overview.txt).
* It allows the model to go beyond relying only on information memorized during its training (ai_overview.txt).
* The system retrieves relevant passages from a trusted collection before the model generates an answer (rag_applications.txt).


In [14]:
def rag_answer(query, provider="groq", show_sources=True):
    context_chunks = retrieve_context(query)

    if not context_chunks:
        return "I don't know based on the provided documents. (No relevant chunks were retrieved.)"

    user_prompt = build_user_prompt(query, context_chunks)

    if provider == "gemini":
        answer = ask_gemini(user_prompt)
    else:
        answer = ask_groq(user_prompt)

    if show_sources:
        sources = sorted({f"{c['source']} #{c['chunk_index']}" for c in context_chunks})
        answer += "\n\nRetrieved from: " + ", ".join(sources)

    return answer

# Pick the provider you have a key for
default_provider = "groq" if GROQ_API_KEY else "gemini"
print("Using provider:", default_provider)


Using provider: groq


In [15]:
q1 = "What is deep learning?"
print("Question:", q1, "\n")
print(rag_answer(q1, provider=default_provider))


Question: What is deep learning? 

- Deep learning extends machine learning by using neural networks with many layers. [ai_overview.txt]  
- These layered networks automatically discover complex patterns in large datasets. [ai_overview.txt]  
- It enables systems to learn from data rather than relying on hardcoded rules. [ai_overview.txt]

Retrieved from: ai_overview.txt #1, ai_overview.txt #2, ai_overview.txt #3, ai_overview.txt #4


In [16]:
q2 = "How do embeddings and vector databases help RAG find useful information?"
print("Question:", q2, "\n")
print(rag_answer(q2, provider=default_provider))


Question: How do embeddings and vector databases help RAG find useful information? 

- Embeddings transform both search queries and documents into comparable vector representations, enabling semantic matching beyond keyword overlap. [embeddings_in_practice.txt]  
- Vector databases (e.g., ChromaDB) store these embeddings and perform rapid similarity searches across large corpora, returning the most relevant documents. [embeddings_in_practice.txt]  
- Retrieval‑Augmented Generation (RAG) leverages this retrieval step to feed the language model with contextually relevant passages, improving answer accuracy. [ai_overview.txt]

Retrieved from: ai_overview.txt #6, embeddings_in_practice.txt #0, embeddings_in_practice.txt #1, embeddings_in_practice.txt #4


In [17]:
q3 = "What is the best recipe for chocolate cake?"
print("Question:", q3, "\n")
print(rag_answer(q3, provider=default_provider))


Question: What is the best recipe for chocolate cake? 

I don't know based on the provided documents. (No relevant chunks were retrieved.)


In [18]:
compare_q = "Why is RAG more reliable than asking a language model directly?"
print("Question:", compare_q, "\n")

compare_chunks = retrieve_context(compare_q)
compare_prompt = build_user_prompt(compare_q, compare_chunks)

print("=" * 25, "GROQ", "=" * 25)
print(ask_groq(compare_prompt))

print("\n" + "=" * 24, "GEMINI", "=" * 24)
print(ask_gemini(compare_prompt))


Question: Why is RAG more reliable than asking a language model directly? 

========================= GROQ =========================
- RAG first retrieves relevant passages from a trusted collection, so the model uses real data instead of only memorized knowledge. (Source: rag_applications.txt, ai_overview.txt)  
- The retrieved passages are up‑to‑date, ensuring the answer reflects current information rather than outdated training data. (Source: ai_overview.txt)  
- Grounding the generation in retrieved content reduces hallucinations and increases accuracy. (Source: rag_applications.txt, embeddings_in_practice.txt)

======================== GEMINI ========================
* RAG uses a trusted collection of information instead of relying solely on what a language model memorized during training (rag_applications.txt, ai_overview.txt).
* It retrieves relevant and up-to-date information from a document collection to provide context for the model (ai_overview.txt).
* This technique allows 

In [ ]:
while True:
    user_question = input("Ask a question (or press Enter to quit): ").strip()
    if not user_question:
        print("Goodbye!")
        break
    print("\n" + rag_answer(user_question, provider=default_provider) + "\n")


Ask a question (or press Enter to quit): quit

I don't know based on the provided documents. (No relevant chunks were retrieved.)



In [20]:
q_app = "What are practical applications of RAG?"
app_chunks = retrieve_context(q_app)
print("Question:", q_app)
print("Retrieved chunks:", len(app_chunks))
for chunk in app_chunks:
    print(f"\nSource: {chunk['source']} | similarity: {chunk['similarity']}")
    print(chunk["text"])


Question: What are practical applications of RAG?
Retrieved chunks: 2

Source: rag_applications.txt | similarity: 0.6589
RAG is useful for internal knowledge assistants, customer support chatbots, and document question-answering tools. It retrieves relevant passages from a trusted collection before a language model writes an answer.

Source: ai_overview.txt | similarity: 0.5011
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG


In [21]:
temperature_question = "How do embeddings and vector databases help RAG?"
temperature_chunks = retrieve_context(temperature_question)
temperature_prompt = build_user_prompt(temperature_question, temperature_chunks)

for temp in (0.1, 1.0):
    RAG_TEMPERATURE = temp
    print(f"\n--- Temperature: {temp} ---")
    if default_provider == "gemini":
        print(ask_gemini(temperature_prompt))
    else:
        print(ask_groq(temperature_prompt))

RAG_TEMPERATURE = 0.1



--- Temperature: 0.1 ---
- Embeddings convert documents into vector representations that capture semantic meaning, enabling comparison of content beyond keyword matching. (embeddings_in_practice.txt)  
- Vector databases such as ChromaDB store these embeddings and perform fast similarity search across thousands or millions of documents, quickly retrieving the most relevant passages. (embeddings_in_practice.txt)  
- RAG leverages this retrieval step—using the search system to fetch trusted passages before the language model generates an answer—so the model can base its response on up‑to‑date, context‑specific information rather than only its training data. (rag_applications.txt, ai_overview.txt)

--- Temperature: 1.0 ---
- Embeddings convert text into high‑dimensional vectors that capture semantic content, enabling quantitative similarity comparisons. (embeddings_in_practice.txt)  
- Vector databases (e.g., ChromaDB) index these vectors and provide fast nearest‑neighbour search across 

## Practice Exercise 3 —

In [22]:
print(SYSTEM_PROMPT)


You are a helpful assistant that answers using ONLY the provided context. If the answer is absent, reply exactly: "I don't know based on the provided documents." For questions answered from context, format the response as exactly 3 bullet points. Keep each bullet concise and cite the source file name(s) used.


In [23]:
test_query = "How do embeddings and vector databases help RAG find useful information?"

for n in (4, 1):
    chunks = retrieve_context(test_query, n_results=n)
    print(f"\n--- n_results={n}; chunks passing threshold={len(chunks)} ---")
    for chunk in chunks:
        print(f"[{chunk['source']} #{chunk['chunk_index']}, similarity={chunk['similarity']}]")
        print(chunk["text"])
    if not chunks:
        print("No chunks passed the similarity threshold.")



--- n_results=4; chunks passing threshold=4 ---
[embeddings_in_practice.txt #0, similarity=0.7577]
Embeddings and Vector Search in Practice
[embeddings_in_practice.txt #4, similarity=0.6304]
Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once.
[embeddings_in_practice.txt #1, similarity=0.5881]
Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
[ai_overview.txt #6, similarity=0.4318]
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG

--- n_results=1; chunks passing threshold=1 ---
[embeddings_in_practice.txt #0, similarity=0.7577]
Embeddings and Vector Search in Practice
